In [223]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

C:\Users\isabe\AppData\Local\Temp\ipykernel_39756\3777615979.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [224]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [225]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("C:/Users/isabe/OneDrive/Документы/ironhack/LEaRNGIT/lab 33/lab-natural-language-processing/data/kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


### Let's divide the training and test set into two partitions

In [226]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(data['text'], data['label'], test_size=0.2, random_state=42)

X_train.head()


29     ----------- REGARDS, MR NELSON SMITH.KINDLY RE...
535    I have not been able to reach oscar this am. W...
695    ; Huma Abedin B6I'm checking with Pat on the 5...
557    I can have it announced here on Monday - can't...
836        BANK OF AFRICAAGENCE SAN PEDRO14 BP 1210 S...
Name: text, dtype: object

## Data Preprocessing

In [227]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [237]:
import re
import string
from bs4 import BeautifulSoup
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

# Function to clean HTML and remove JavaScript/CSS
def clean_html(text):
    text = re.sub(r'<script.*?>.*?</script>', '', text, flags=re.DOTALL)  # Remove <script>...</script>
    text = re.sub(r'<style.*?>.*?</style>', '', text, flags=re.DOTALL)  # Remove <style>...</style>
    text = re.sub(r'<!--.*?-->', '', text, flags=re.DOTALL)  # Remove HTML comments
    soup = BeautifulSoup(text, "html.parser")
    text = soup.get_text(separator=" ")  # Extract plain text
    return text

#  Function for full text preprocessing
def preprocess_text(text):
    text = clean_html(text)  # Clean HTML
    text = text.lower()  # Convert to lowercase

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    #  Remove all special characters
    text = text.translate(str.maketrans("", "", string.punctuation))

    #  Remove all single characters (e.g., "a", "b", "c")
    text = re.sub(r'\s+[a-zA-Z]\s+', ' ', text)

    #  Remove single characters at the start of words
    text = re.sub(r'\b[a-zA-Z]\b', '', text)

    #  Replace multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text).strip()

    # Remove 'b' prefix (in case texts are byte strings)
    text = re.sub(r'\bb\s+', '', text)

    # Tokenization and stopword removal
    tokens = text.split()
    stop_words = set(stopwords.words("english"))
    tokens = [word for word in tokens if word not in stop_words]

    # Apply stemming
    stemmer = SnowballStemmer("english")
    tokens = [stemmer.stem(word) for word in tokens]

    return " ".join(tokens)

# Apply the function to train and test data
X_train = X_train.apply(preprocess_text)
X_test = X_test.apply(preprocess_text)
X_train.head()



29     egard elson mithkind epli rivat mail ddressnel...
535                    ave ot een le scar uppo end eceiv
695    uma edin im heck ith ill ork ith ack ake estal...
557                        ave nnounc ere onday ant oday
836    ank fricaag edro edro ote ivoirewest frica ear...
Name: text, dtype: object

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

## Now let's work on removing stopwords
Remove the stopwords.

In [230]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    return ' '.join([word for word in text.split() if word.lower() not in stop_words])

X_train = X_train.apply(remove_stopwords)
X_test = X_test.apply(remove_stopwords)

print(X_train.head())

29     egards r elson mithkindly eply e n rivate mail...
535              ave ot een le scar e upposed end eceive
695    uma edin im hecking ith n ill ork ith ack ake ...
557                    ave nnounced ere n onday ant oday
836    ank f fricaagence edro p edro ote ivoirewest f...
Name: text, dtype: object


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\isabe\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [231]:
import nltk

nltk.download("punkt")  # Word tokenizer
nltk.download("wordnet")  # Lemmatization
nltk.download("stopwords")  # Stopwords
nltk.download("omw-1.4")  # WordNet lexical database

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\isabe\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\isabe\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\isabe\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\isabe\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [232]:
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    return' '.join([lemmatizer.lemmatize(word) for word in text.split()])

X_train = X_train.apply(lemmatize_text)

X_test = X_test.apply(lemmatize_text)

X_train.head()


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\isabe\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


29     egards r elson mithkindly eply e n rivate mail...
535              ave ot een le scar e upposed end eceive
695    uma edin im hecking ith n ill ork ith ack ake ...
557                    ave nnounced ere n onday ant oday
836    ank f fricaagence edro p edro ote ivoirewest f...
Name: text, dtype: object

## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [233]:
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
import re

# Configure the CountVectorizer
vectorizer = CountVectorizer(
    stop_words='english',  # Remove English stop words
    lowercase=True,        # Convert text to lowercase
    token_pattern=r'\b[a-zA-Z]+\b',  # Only consider words with letters
    max_features=1000      # Limit the number of features
)

# Transform the text into a Bag of Words (BoW) matrix
X_train_bow = vectorizer.fit_transform(X_train)
feature_names = vectorizer.get_feature_names_out()

# Separate ham and spam messages
ham_indices = y_train == 0  # Assuming ham is labeled as 0
spam_indices = y_train == 1  # Assuming spam is labeled as 1

X_train_ham_bow = X_train_bow[ham_indices]  # BoW for ham messages
X_train_spam_bow = X_train_bow[spam_indices]  # BoW for spam messages

# Function to get the top N words
def get_top_words(bow_features, feature_names, n=10):
    # Sum the word counts along axis 0 (columns)
    word_counts = np.asarray(bow_features.sum(axis=0)).flatten()
    # Create a list of (word, count) tuples
    word_counts = list(zip(feature_names, word_counts))
    # Sort the list by count in descending order and select the top N words
    sorted_words = sorted(word_counts, key=lambda x: x[1], reverse=True)[:n]
    return sorted_words

# Get and print the top 10 words for ham and spam
top_ham_words = get_top_words(X_train_ham_bow, feature_names, n=10)
top_spam_words = get_top_words(X_train_spam_bow, feature_names, n=10)

print("Top 10 words in HAM messages:")
for word, count in top_ham_words:
    print(f"{word}: {count}")

print("\nTop 10 words in SPAM messages:")
for word, count in top_spam_words:
    print(f"{word}: {count}")

Top 10 words in HAM messages:
n: 917
f: 775
nd: 761
e: 526
hat: 420
ou: 248
ith: 206
ut: 198
ill: 195
r: 194

Top 10 words in SPAM messages:
f: 4310
n: 3657
e: 3346
nd: 3342
ou: 2467
ill: 1512
hat: 1455
ith: 1043
ave: 805
oney: 795


## Extra features

In [234]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€","$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

NameError: name 'data_train' is not defined

In [214]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).

# Convert X_train and X_test to DataFrames
X_train = pd.DataFrame(X_train, columns=['text'])
X_test = pd.DataFrame(X_test, columns=['text'])

# List of currency symbols and suspicious words
money_symbol_list = "|".join(["euro", "dollar", "pound", "€", "$"])
suspicious_words = "|".join(["free", "cheap", "sex", "money", "account", "bank", "fund", "transfer", "transaction", "win", "deposit", "password"])

# Add indicators to the training DataFrame
X_train['money_mark'] = X_train['text'].str.contains(money_symbol_list, case=False, regex=True) * 1
X_train['suspicious_words'] = X_train['text'].str.contains(suspicious_words, case=False, regex=True) * 1
X_train['text_len'] = X_train['text'].apply(lambda x: len(x))

# Add indicators to the test DataFrame
X_test['money_mark'] = X_test['text'].str.contains(money_symbol_list, case=False, regex=True) * 1
X_test['suspicious_words'] = X_test['text'].str.contains(suspicious_words, case=False, regex=True) * 1
X_test['text_len'] = X_test['text'].apply(lambda x: len(x))

X_train.head()

,text,money_mark,suspicious_words,text_len
29,gards lson ithkindly ply ivate ail dressnelson...,1,0,59
535,en e ach car e pposed nd e n ceive,1,0,34
695,din ecking th e rk th ck ke stalso r u llow mo...,1,0,54
557,n nounced nday nt day,1,0,21
836,nk ricaagence n dro n dro te voirewest rica ar...,1,0,848


## How would you create a Bag of Words with the CountVectorizer method?

In [215]:


from sklearn.feature_extraction.text import CountVectorizer

# Configure the CountVectorizer

vectorizer = CountVectorizer(
    stop_words='english',  # Remove English stop words
    lowercase=True,        # Convert text to lowercase
    token_pattern=r'\b[a-zA-Z]+\b',  # Only consider words with letters
    max_features=1000      # Limit the number of features
)

# Transform the text into a Bag of Words (BoW) matrix

X_train_bow = vectorizer.fit_transform(X_train['text'])
feature_names = vectorizer.get_feature_names_out()

print("BoW matrix shape:", X_train_bow.shape)

print("Top 10 most frequent words:")

top_words = get_top_words(X_train_bow, feature_names, n=10)

for word, count in top_words:
    print(f"{word}: {count}")
    print()

BoW matrix shape: (800, 1000)
Top 10 most frequent words:
e: 8670

r: 2835

u: 2773

ur: 1898

th: 1383

n: 1068

nd: 886

om: 837

ney: 799

w: 703



## TD-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [216]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Configure the TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words='english',  # Remove English stop words
    lowercase=True,        # Convert text to lowercase
    token_pattern=r'\b[a-zA-Z]+\b',  # Only consider words with letters
    max_features=1000      # Limit the number of features
)

# Vectorize the text into a TF-IDF matrix

X_train_tfidf = vectorizer.fit_transform(X_train['text'])
feature_names = vectorizer.get_feature_names_out()

print("TF-IDF matrix shape:", X_train_tfidf.shape)

print("Top 10 most frequent words:")

top_words = get_top_words(X_train_tfidf, feature_names, n=10)

for word, count in top_words:
    print(f"{word}: {count}")
    print()

TF-IDF matrix shape: (800, 1000)
Top 10 most frequent words:
e: 201.45962802853688

u: 85.37241495734068

r: 85.17043684831577

ur: 59.83282458700939

th: 42.078635723778234

n: 40.07708854443676

w: 32.18165298815929

om: 29.788533924380882

nd: 29.615559046376

st: 28.095466626762416



### Extra Task (optional) - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

Use a MultinimialNB with default parameters.

Your task is to find the **best feature representation**.

You can work with teams of two persons (recommended).

In [ ]:
# Your code